# Law data vector database creation

## Database installation

Create a vector database using Milvus db with docker.

```bash
curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o standalone_embed.sh

bash standalone_embed.sh start
```

### Useful commands

```bash
bash standalone_embed.sh upgrade

bash standalone_embed.sh stop

bash standalone_embed.sh delete
```

For extra info check the [docs](https://milvus.io/docs/manage_databases.md).

### Download and install the db server

In [18]:
!curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o standalone_embed.sh

In [19]:
!bash standalone_embed.sh start

[sudo] password for leospairani: 
sudo: a password is required
^C


```bash
Wait for Milvus Starting...
Start successfully.
To change the default Milvus configuration, add your settings to the user.yaml file and then restart the service.
```


### Install the Python SDK

In [21]:
!uv add pymilvus

Resolved 256 packages in 1.47s                                       
⠙ Preparing packages... (0/1)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/1)--------------     0 B/252.77 KiB          
⠙ Preparing packages... (0/1)-------------- 16.00 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 32.00 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 48.00 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 60.90 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 76.90 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 92.90 KiB/252.77 KiB        
⠙ Preparing packages... (0/1)-------------- 108.90 KiB/252.77 KiB       
⠙ Preparing packages... (0/1)-------------- 124.90 KiB/252.77 KiB       
⠙ Preparing packages... (0/1)m------------- 140.90 KiB/252.77 KiB       
⠙ Preparing packages... (0/1)----------

### Create the database

In [ ]:
from pymilvus import MilvusClient

def connect_milvus():
    client = MilvusClient(
        uri="http://localhost:19530",
        token="root:Milvus"
    )
    return client

client = connect_milvus()

print(client.list_databases())

print(client.describe_database(
    db_name="default"
))

['default']
{'name': 'default'}


In [ ]:
# Create database
client.create_database("tesis_MCD_v1")

In [158]:
print(client.list_databases())

['default', 'tesis_MCD_v1']


Useful commands:

- Switch from database without disconnecting (not for REST api, only SDK)

```python
client.use_database(
    db_name="my_database_2"
)
```


- Drop databases

```python
client.drop_database(
    db_name="my_database_2"
)
```

## Create initial collections

https://milvus.io/docs/manage-collections.md

- Code to drop a collection if exists

```python
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)
```

#### Build schema


In [245]:
from pymilvus import DataType

embedding_dimension = 768 # Based on embedding generation model

def create_collection():
    try:
        client = connect_milvus()

        client.use_database(
            db_name="tesis_MCD_v1"
        )

        # 3.1. Create schema
        schema = MilvusClient.create_schema(
            auto_id=False,
            enable_dynamic_field=False,
        )

        # 3.2. Add fields to schema
        schema.add_field(field_name="id_parte", datatype=DataType.VARCHAR, max_length=32, is_primary=True, auto_id=False)
        schema.add_field(field_name="codigo_norma", datatype=DataType.VARCHAR, max_length=32)
        schema.add_field(field_name="tipo_parte", datatype=DataType.VARCHAR, max_length=20)
        schema.add_field(field_name="numero_parte", datatype=DataType.INT16, nullable=True)
        schema.add_field(field_name="id_padre", datatype=DataType.VARCHAR, max_length=32, nullable=True)
        schema.add_field(field_name="titulo_parte", datatype=DataType.VARCHAR, max_length=256, nullable=True)
        schema.add_field(field_name="ultimo_estado", datatype=DataType.VARCHAR, max_length=20, nullable=True)
        schema.add_field(field_name="contenido_texto", datatype=DataType.VARCHAR, max_length=8192, nullable=True)

        # Embedding fields
        schema.add_field(field_name="vector_contenido_texto", datatype=DataType.FLOAT_VECTOR, dim=embedding_dimension)

        # # 3.3. Prepare index parameters
        index_params = client.prepare_index_params()

        # # 3.4. Add indexes
        # index_params.add_index(
        #     field_name="id_padre",
        #     index_type="AUTOINDEX"
        # )

        index_params.add_index(
            field_name="vector_contenido_texto", 
            index_type="AUTOINDEX",
            metric_type="COSINE"
        )

        # 3.5. Create a collection with the index loaded simultaneously
        # client.create_collection(
        #     collection_name="customized_setup_1",
        #     schema=schema,
            # index_params=index_params
        # )

        # res = client.get_load_state(
        #     collection_name="customized_setup_1"
        # )

        # print(res)
        # 3.6. Create a collection and index it separately

        # embedding_dim <-- Create a test embedding to set the dimension

        client.create_collection(
            collection_name="codigo_de_mineria_v1",
            # dimension=embedding_dim,
            schema=schema,
            index_params=index_params,
            consistency_level="Strong",
        )

        res = client.get_load_state(
            collection_name="codigo_de_mineria_v1"
        )

        print(res)
    except Exception as e:
        print("Exception:", e)
    finally:
        client.close()


In [ ]:
# client = connect_milvus()
# client.use_database("tesis_MCD_v1")
# client.drop_collection("codigo_de_mineria_v1")
# client.close()

## Load and format law data

Load json data and load it into tuples with the format

`(codigo_norma, id_parte, id_padre, tipo_parte, numero_parte, titulo_parte, ultimo_estado, contenido_texto)`


- #TODO mejoras pendientes:
    - Se puede agregar un campo `id_modificado_por`

In [236]:
from pathlib import Path
import json
import hashlib

BASE_DIR = Path.cwd().parent
DATA_DIR_PATH = BASE_DIR.joinpath("data", "law_data")
DATA_FILE_PATH = DATA_DIR_PATH.joinpath("codigo_de_mineria_actualizado_2025_08_02.json")

with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

codigo_norma = "Código de Minería Argentino"
result = []

def generate_id(text:str):
    return hashlib.md5(text.encode()).hexdigest()

def extract_part_data(codigo_norma:str, parte:dict, id_parte:str, id_padre:str|None=None)-> tuple[str|None]:
    return {
        "codigo_norma": codigo_norma,
        "id_parte": id_parte,
        "id_padre": id_padre,
        "tipo_parte": parte.get("tipo_parte", None),
        "numero_parte": parte.get("numero_parte", None),
        "titulo_parte": parte.get("titulo_parte", None),
        "ultimo_estado": parte.get("ultimo_estado", None),
        "contenido_texto": parte.get("contenido_texto", None),
        "vector_contenido_texto": None,
    }


def process_parts(codigo_norma:str, data: list[dict], id_padre:str|None=None):
    for entity in data:
        s = f"{codigo_norma},{entity.get('tipo_parte', '')},{entity.get('numero_parte', '')},{entity.get('titulo_parte', '')}"
        id_parte = generate_id(s)
        result.append(
            extract_part_data(
                codigo_norma,
                entity,
                id_parte,
                id_padre,
            )
        )
        if "partes" in entity.keys():
            if entity.get("partes"):
                process_parts(codigo_norma, entity.get("partes"), id_parte)

process_parts(codigo_norma, data=data["partes"])
result

[{'codigo_norma': 'Código de Minería Argentino',
  'id_parte': 'ebad6798643112a64729d7cc7a1abf2c',
  'id_padre': None,
  'tipo_parte': 'título',
  'numero_parte': 1,
  'titulo_parte': 'De las minas y su dominio',
  'ultimo_estado': 'en vigor',
  'contenido_texto': None,
  'vector_contenido_texto': None},
 {'codigo_norma': 'Código de Minería Argentino',
  'id_parte': '78020a7e760ce39a4591da47093358d7',
  'id_padre': 'ebad6798643112a64729d7cc7a1abf2c',
  'tipo_parte': 'artículo',
  'numero_parte': 1,
  'titulo_parte': None,
  'ultimo_estado': 'en vigor',
  'contenido_texto': 'El Código de Minería rige los derechos, obligaciones y procedimientos referentes a la adquisición, explotación y aprovechamiento de las sustancias minerales.',
  'vector_contenido_texto': None},
 {'codigo_norma': 'Código de Minería Argentino',
  'id_parte': 'e951a8bcd2ec577d42cf711ce64c6fb7',
  'id_padre': 'ebad6798643112a64729d7cc7a1abf2c',
  'tipo_parte': 'acápite',
  'numero_parte': 1,
  'titulo_parte': 'Clasific

In [237]:
max_len = 0
art_nro = None
tipos_partes = set()
for item in result:
    tipo = item.get("tipo_parte")
    nro = item.get("numero_parte")
    texto = item.get("contenido_texto")
    if tipo:
        tipos_partes.add(tipo)
    if texto:
        text_len = len(texto)
        if text_len > max_len:
            max_len = text_len
            art_nro = nro

print(f"Art N°{art_nro}: {max_len}")
print(f"Los tipo de partes son: {tipos_partes}")

Art N°346: 3761
Los tipo de partes son: {'artículo', 'apéndice', 'título', 'sección', 'acápite'}


## Create embeddings

- Selected model: [hiiamsid/sentence_similarity_spanish_es](https://huggingface.co/hiiamsid/sentence_similarity_spanish_es)
    - Embedding dimension: `768`
- **Chunking**: the documents are split on articles. In this case the longest one has 3761 characters. As a first step no chunking will be added.
- **Tokenization**: it is handled by the model

In [238]:
from sentence_transformers import SentenceTransformer
import copy
import numpy as np

embedding_model_name = "hiiamsid/sentence_similarity_spanish_es"

embedding_model = SentenceTransformer(embedding_model_name)

null_vector = np.zeros((embedding_dimension,), dtype=np.float32)

def generate_chunks(text:str):
    """
    Split the text into chunks of a specified maximum length.
    """
    pass

def tokenize(text:str):
    pass

def generate_embedding(text: str, model) -> list[float]:
    return model.encode(text)


result_with_embeddings = []

for item in result:
    if item.get("contenido_texto"):
        item["vector_contenido_texto"] = generate_embedding(item.get("contenido_texto"), embedding_model)
    else:
        item["vector_contenido_texto"] = null_vector
    result_with_embeddings.append(copy.deepcopy(item))
result_with_embeddings

[{'codigo_norma': 'Código de Minería Argentino',
  'id_parte': 'ebad6798643112a64729d7cc7a1abf2c',
  'id_padre': None,
  'tipo_parte': 'título',
  'numero_parte': 1,
  'titulo_parte': 'De las minas y su dominio',
  'ultimo_estado': 'en vigor',
  'contenido_texto': None,
  'vector_contenido_texto': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0

In [239]:
result_with_embeddings[1]["vector_contenido_texto"].shape

(768,)

## Populate DB

In [ ]:
database_name = "tesis_MCD_v1"
collection_name = "codigo_de_mineria_v1"
answer_vector_field = "vector_contenido_texto"

try:
    create_collection()
    client = connect_milvus()
    client.use_database(
        db_name=database_name
    )
    client.load_collection(collection_name)
    res = client.get_load_state(
        collection_name=collection_name
    )
    print(res)
    res = client.insert(
        collection_name=collection_name,
        data=result_with_embeddings
    )
    print(res)
    print(client.get_collection_stats(collection_name))
except Exception as e:
    print("Exception:", e)
    client.drop_collection(collection_name)
finally:
    client.close()

{'state': <LoadState: Loaded>}
{'state': <LoadState: Loaded>}
{'insert_count': 482, 'ids': ['ebad6798643112a64729d7cc7a1abf2c', '78020a7e760ce39a4591da47093358d7', 'e951a8bcd2ec577d42cf711ce64c6fb7', '845967b6821b6b963355ac57c9acbdde', '63cd4d88a28c3de1e57aa737c6ae495d', 'b0e659f5cf64a11ccd3ca31e56f8c56f', '60dfbac337127c856b521b7a4d275a34', '7d4a10eefb050b2abaadc2046b0e2e2b', '4f349b3799fb5e95b30eea8e2d13ca9a', '7e30095ca21fb10aa2934c5f1bc0e873', 'ec7955314f93bd799c6c695e2c1cda72', '80aea83c555cd87853b29cf5ddabf895', '15fae8d5ad2c466361dc76ce3f016c3a', 'cff27d65382d7a3bc237b279ecd0ad4d', 'bedf2671d4f34be93a52c2253f634015', '4d442cd021677333a58832d245596294', '29e37cf4b97293026184d5cef7779c8f', '47febf76b128dec17c5858ad982c06f3', '2b6d4b6917707453cf28534cd9a2363b', '29bb34e2f4d79e186d96f1d1da046d8e', '3cf7a7b3838dc7b3f9c98ba3060190aa', '74513e0e2a98e5e636fcfa7b8ae77761', '8052e97dce5d023ad033ecb592c6d4b3', '3d2627715c5978cb1ed27c7cf81c6c22', '8900371ff9cc8538be182a9a20fe3a5f', 'a46500a

In [254]:
client = connect_milvus()
client.use_database(database_name)
print(client.list_collections())
print(client.get_collection_stats(collection_name=collection_name))
client.close()

['codigo_de_mineria_v1']
{'row_count': 0}


In [258]:
client = connect_milvus()
client.use_database(database_name)
res = client.get(
    collection_name=collection_name,
    ids=["ebad6798643112a64729d7cc7a1abf2c", "78020a7e760ce39a4591da47093358d7"],
    output_fields=["id_parte", "contenido_texto", "vector_contenido_texto"]
)

print(res)
client.close()

data: ["{'id_parte': '78020a7e760ce39a4591da47093358d7', 'contenido_texto': 'Las minas de petróleo e hidrocarburos fluidos son bienes del dominio privado de la Nación o de las provincias, según el territorio en que se encuentren.', 'vector_contenido_texto': [-0.24501162767410278, -0.5617707967758179, -0.868441641330719, 0.04099174961447716, 1.003210425376892, 0.4362984299659729, 1.202682375907898, 0.28241774439811707, 0.1131332740187645, -0.056747157126665115, -0.48038578033447266, -0.8037906289100647, -0.15525299310684204, 0.46606338024139404, -0.169274240732193, -0.16620339453220367, 0.2900070250034332, 0.04672738537192345, -0.152260884642601, -0.1401180624961853, -1.2984206676483154, 1.205493688583374, 0.2672310471534729, -0.2592608630657196, -0.14466743171215057, -0.11465469747781754, 0.6772140860557556, -0.11481547355651855, -0.8171981573104858, -0.10074026137590408, 0.3994868993759155, -0.5620229840278625, -1.0737462043762207, 0.6148869395256042, -0.1595810502767563, 0.6084326505

## Test retrieving data

#TODO: search variables to test:

- Metric Types: https://milvus.io/docs/metric.md
    - Euclidean distance (L2)
    - Inner Product (IP)
    - Cosine Similarity (COSINE)
    - JACCARD
    - HAMMING
    - BM25 (specifically designed for full text search on sparse vectors).

- Idex type: https://milvus.io/docs/index-explained.md
    - HNSW
    - DISKANN
    - FLAT
    - IVF_FLAT
    - IVF_SQ8
    - IVF_PQ
    - SCANN
    - GPU_IFV_FLAT
    - GPU_IVF_PQ
    - GPU_CAGRA
    - GPU_BRUTE_FORCE
    - SPARSE_INVERTED_INDEX
    - BIN_FLAT
    - BIN_IVF_FLAT

### Text search

In [265]:
query_text = "La señora Ana López ha recibido en usufructo por un plazo de 30 años un predio rural que pertenece a la familia Fernández. Dentro de este predio se encuentra una cantera de piedra que no se encuentra en actual explotación, así como una mina de oro en explotación registrada a nombre de los propietarios. A los pocos años, se concede otra mina dentro del mismo predio a una empresa minera, y durante los trabajos, se producen destrucciones de cultivos y deterioro del suelo. De acuerdo al Código de Minería Argentino: ¿Puede la señora Ana López explotar por sí o por terceros la cantera y la mina de oro, y en qué condiciones?"

#- Artículo 338
#- Artículo 339
#- Artículo 342

question_vector = generate_embedding(query_text, embedding_model)


In [ ]:
# 4. Single vector search
output_fields = [
    "codigo_norma",
    "id_parte",
    "id_padre",
    "tipo_parte",
    "numero_parte",
    "titulo_parte",
    "ultimo_estado",
    "contenido_texto",
]

client = connect_milvus()
client.use_database(database_name)
results = client.search(
    collection_name=collection_name,
    anns_field=answer_vector_field,
    data=[question_vector],
    limit=25,
    search_params={"metric_type": "COSINE"},
    output_fields=output_fields
)

for i, result in enumerate(results):
    for j, res in enumerate(result):
        print(i, j, res["distance"], res["entity"].get("numero_parte", None))
client.close()

0 0 0.7745012044906616 24
0 1 0.7115315794944763 338
0 2 0.7042615413665771 12
0 3 0.699307918548584 350
0 4 0.6923577785491943 222
0 5 0.6898420453071594 158
0 6 0.6807446479797363 319
0 7 0.6798427104949951 14
0 8 0.6745507717132568 340
0 9 0.6729925274848938 342
0 10 0.6666476130485535 201
0 11 0.6629376411437988 44
0 12 0.6626238226890564 339
